# 04 - LangChain com memória

O modelo não guarda nada entre uma chamada e outra. Quando ele parece lembrar, é porque o histórico foi enviado de novo junto com a pergunta.

Em LangChain puro (`langchain-core`) dá para fazer isso sem frameworks extras:

1. Mostrar uma conversa **sem memória**.
2. Guardar o histórico com `InMemoryChatMessageHistory` e injetá-lo no prompt com `MessagesPlaceholder`, usando `RunnableWithMessageHistory`.

As configurações de chave e modelo continuam vindo do `.env` na raiz do projeto.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 1. Sem memória

Cada `invoke` é independente. A segunda pergunta não contém nada da primeira, então o modelo não tem como saber o nome.

In [2]:
cadeia_sem_memoria = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "Responda em português, em uma ou duas frases."),
            ("human", "{mensagem}"),
        ]
    )
    | llm
    | StrOutputParser()
)

primeira = cadeia_sem_memoria.invoke({"mensagem": "Oi, meu nome é Leonardo e estou estudando LangChain."})
segunda = cadeia_sem_memoria.invoke({"mensagem": "Qual é o meu nome e o que eu estou estudando?"})

display(Markdown(f"**Você:** Oi, meu nome é Leonardo e estou estudando LangChain.\n\n**Modelo:** {primeira}"))
display(Markdown(f"**Você:** Qual é o meu nome e o que eu estou estudando?\n\n**Modelo:** {segunda}"))

**Você:** Oi, meu nome é Leonardo e estou estudando LangChain.

**Modelo:** Oi Leonardo, prazer — LangChain é uma biblioteca para conectar modelos de linguagem com ferramentas e fluxos de dados; quer que eu explique os conceitos principais ou mostre um exemplo prático em código?

**Você:** Qual é o meu nome e o que eu estou estudando?

**Modelo:** Não sei seu nome nem o que você está estudando — não tenho acesso a informações pessoais. Se quiser, diga-me seu nome e o que estuda que eu respondo.

## 2. Com memória

Agora o prompt tem um `MessagesPlaceholder` para o histórico. O `RunnableWithMessageHistory` lê as mensagens anteriores da sessão, injeta no prompt e salva a nova troca ao final de cada chamada.

In [3]:
prompt_com_memoria = ChatPromptTemplate.from_messages(
    [
        ("system", "Você é um assistente prestativo. Responda em português, em uma ou duas frases."),
        MessagesPlaceholder("historico"),
        ("human", "{mensagem}"),
    ]
)

armazenamento = {}


def obter_historico(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in armazenamento:
        armazenamento[session_id] = InMemoryChatMessageHistory()
    return armazenamento[session_id]


cadeia_com_memoria = RunnableWithMessageHistory(
    prompt_com_memoria | llm | StrOutputParser(),
    obter_historico,
    input_messages_key="mensagem",
    history_messages_key="historico",
)

config = {"configurable": {"session_id": "aula-1"}}

/home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
perguntas = [
    "Oi, meu nome é Leonardo e estou estudando LangChain.",
    "Qual é o meu nome e o que eu estou estudando?",
]

for pergunta in perguntas:
    resposta = cadeia_com_memoria.invoke({"mensagem": pergunta}, config=config)
    display(Markdown(f"**Você:** {pergunta}\n\n**Modelo:** {resposta}"))

/tmp/ipykernel_148999/2197282070.py:14: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  armazenamento[session_id] = InMemoryChatMessageHistory()


**Você:** Oi, meu nome é Leonardo e estou estudando LangChain.

**Modelo:** Oi Leonardo, prazer — posso te ajudar com LangChain (conceitos, exemplos em Python, integração com modelos, chains e agentes, deploy). O que você quer aprender ou qual problema quer resolver?

**Você:** Qual é o meu nome e o que eu estou estudando?

**Modelo:** Seu nome é Leonardo e você está estudando LangChain. Posso ajudar com conceitos, exemplos em Python ou integração — o que você prefere?

Agora o modelo responde com o nome e o assunto, porque o histórico da sessão `aula-1` foi enviado junto.

Conferindo o que está guardado:

In [5]:
for mensagem in armazenamento["aula-1"].messages:
    print(f"*{mensagem.type}*: {mensagem.content}")

*human*: Oi, meu nome é Leonardo e estou estudando LangChain.
*ai*: Oi Leonardo, prazer — posso te ajudar com LangChain (conceitos, exemplos em Python, integração com modelos, chains e agentes, deploy). O que você quer aprender ou qual problema quer resolver?
*human*: Qual é o meu nome e o que eu estou estudando?
*ai*: Seu nome é Leonardo e você está estudando LangChain. Posso ajudar com conceitos, exemplos em Python ou integração — o que você prefere?


## 3. Sessões separadas

A memória é por `session_id`. Uma nova sessão começa vazia, então o modelo não conhece o nome.

In [6]:
config_nova_sessao = {"configurable": {"session_id": "aula-2"}}

resposta = cadeia_com_memoria.invoke(
    {"mensagem": "Qual é o meu nome?"},
    config=config_nova_sessao,
)

display(Markdown(f"**Você (sessão aula-2):** Qual é o meu nome?\n\n**Modelo:** {resposta}"))

**Você (sessão aula-2):** Qual é o meu nome?

**Modelo:** Não sei seu nome — não tenho acesso a informações pessoais a menos que você as compartilhe. Se quiser, diga-me o seu nome e eu o utilizarei nas próximas respostas.

## Observações

- O histórico cresce a cada turno e é enviado inteiro ao modelo, o que aumenta custo e tokens. Para conversas longas, convém limitar ou resumir as mensagens antigas.
- `InMemoryChatMessageHistory` some quando o processo termina. Para persistir, troque por outra implementação de `BaseChatMessageHistory`, como uma que grave em banco de dados.
- As classes antigas de memória (como `ConversationBufferMemory`) estão depreciadas no LangChain 1.x. A abordagem acima é a recomendada.